# RAG Task 1 — Build the Knowledge Base (chunk → embed → store)

This is the **offline** half of RAG — you run it once to load the shelves. Goal: take the security docs in `data/knowledge_base/` and turn them into something searchable *by meaning*, sitting in a Chroma vector store.

The pipeline, which is the whole task:

```
docs  →  chunk  →  embed  →  store in Chroma
```

- **chunk** — slice each doc into small, single-topic passages (retrieval works better on focused pieces than whole files).
- **embed** — turn each chunk into a ~384-number vector that encodes its *meaning* (`sentence-transformers`). Similar meaning → nearby vectors.
- **store** — drop the vectors + their original text into Chroma, which later answers “which chunks are nearest to *this* one?”

Fill each `# TODO`, run it (Shift+Enter), check the output. Ping me to review when the sanity check at the end works.

## Step 0 — Imports
You need three things: `pathlib` (to find the doc files), `SentenceTransformer` (embeddings), and `chromadb` (vector store).

**Hints:**
- `from pathlib import Path`
- `from sentence_transformers import SentenceTransformer`
- `import chromadb`

In [3]:
# TODO: import Path, SentenceTransformer, chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

## Step 1 — Load the docs
Read every `.md` file in `data/knowledge_base/` into memory. End with a list of `(filename, full_text)` pairs — you'll need the filename later as the chunk's *source* (citations!).

**Hints:**
- `kb_dir = Path('../../data/knowledge_base')` (this notebook lives two levels deep under `notebooks/phase7_rag/`).
- `kb_dir.glob('*.md')` yields each file path; `p.read_text()` gives its contents and `p.stem` gives the name without extension.
- Sanity-check: you should load **4** docs. Print how many and the first ~200 chars of one.

In [4]:
# TODO: load every .md in data/knowledge_base into docs = [(name, text), ...]; print how many you got
kb_dir = Path('/Users/rohanb/06_projects/CYBERSECURITY/data/knowledge_base')


In [5]:
kb_dir.glob('*.md')

In [11]:
docs=[]
for p in kb_dir.glob('*.md'):
    name=p.stem
    text=p.read_text()
    docs.append((name, text))
print(len(docs))

4


In [12]:
print(docs)

[('dos_ddos', "# Denial of Service (DoS) and Distributed Denial of Service (DDoS)\n\n## What it is\nA Denial of Service attack tries to make a service unavailable to legitimate users by exhausting a resource — bandwidth, connection tables, CPU, or memory. A DoS comes from a single source; a DDoS (Distributed DoS) comes from many sources at once, often a botnet, which makes it far harder to block by IP. In MITRE ATT&CK this is T1498 (Network Denial of Service). Common variants in the CICIDS2017 dataset include DoS Hulk, DoS GoldenEye, DoS Slowloris, DoS Slowhttptest, and DDoS.\n\n## How it looks on the network\nDoS/DDoS traffic is the opposite shape of a port scan: loud and heavy. Expect a very high volume of packets or connections to a single destination service, sustained over time. Flood variants (Hulk, GoldenEye, DDoS) show huge packet and byte counts. Slow attacks (Slowloris, Slowhttptest) are sneakier — they hold many connections open with trickle-slow requests to exhaust the serv

## Step 2 — Chunk the docs
Slice each document into smaller passages. Our docs are written with `## ` section headings (What it is / How it looks / Why it matters / Playbook), so **splitting on those headings gives you one clean topic per chunk** — a great first chunking strategy.

Build a parallel list for each chunk: the **text**, a unique **id**, and a **source** (which doc it came from). You'll hand all three to Chroma in Step 4.

**Hints:**
- Split a doc on the heading marker: `text.split('\n## ')`. (The first piece is the `# Title` / intro — keep or drop it, your call; just be consistent.)
- Strip whitespace and skip empty pieces.
- A simple unique id: `f"{name}_{i}"` (doc name + chunk index).
- End with three aligned lists (or one list of dicts): `chunk_texts`, `chunk_ids`, `chunk_sources`. Print the total chunk count and eyeball a couple — do they each read as one coherent topic?

In [18]:
# TODO: split each doc into section chunks; build chunk_texts, chunk_ids, chunk_sources

chunk_texts, chunk_ids, chunk_sources = [], [], []
for name, text in docs:
    sections=text.split('\n## ')
    for i, sec in enumerate(sections):
        sec=sec.strip()
        if not sec:
            continue
        chunk_texts.append(sec)
        chunk_ids.append(f'{name}_{i}')
        chunk_sources.append(name)
        
print(len(chunk_ids))
print(chunk_ids[1],chunk_texts[1])
print(chunk_sources)

20
dos_ddos_1 What it is
A Denial of Service attack tries to make a service unavailable to legitimate users by exhausting a resource — bandwidth, connection tables, CPU, or memory. A DoS comes from a single source; a DDoS (Distributed DoS) comes from many sources at once, often a botnet, which makes it far harder to block by IP. In MITRE ATT&CK this is T1498 (Network Denial of Service). Common variants in the CICIDS2017 dataset include DoS Hulk, DoS GoldenEye, DoS Slowloris, DoS Slowhttptest, and DDoS.
['dos_ddos', 'dos_ddos', 'dos_ddos', 'dos_ddos', 'dos_ddos', 'web_and_bruteforce', 'web_and_bruteforce', 'web_and_bruteforce', 'web_and_bruteforce', 'web_and_bruteforce', 'port_scan', 'port_scan', 'port_scan', 'port_scan', 'port_scan', 'botnet', 'botnet', 'botnet', 'botnet', 'botnet']


## Step 3 — Embed the chunks
Turn every chunk's text into a meaning-vector. This is the heart of RAG.

**Hints:**
- Load the model once: `model = SentenceTransformer('all-MiniLM-L6-v2')` — a small, fast, 384-dim model (first run downloads it, then it's cached).
- `embeddings = model.encode(chunk_texts)` embeds the whole list in one call. `.tolist()` converts the numpy array to plain lists (Chroma wants lists).
- Sanity-check the shape: `len(embeddings)` should equal your chunk count, and each vector should be length 384. Print `embeddings[0][:8]` just to see the numbers are real.

In [21]:
# TODO: load SentenceTransformer('all-MiniLM-L6-v2'); embed chunk_texts; check the shape
model=SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunk_texts).tolist()
print(embeddings)
print(len(embeddings))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[[0.0012165792286396027, 0.013795650564134121, 0.04467237368226051, -0.07359126955270767, -0.03643917292356491, -0.0285458043217659, 0.08039540797472, 0.0043242694810032845, 0.0829034373164177, 0.041284188628196716, 0.030743790790438652, 0.05510864779353142, 0.08962489664554596, -0.0364597849547863, 0.05012323707342148, -0.03437928855419159, 0.07836827635765076, -0.03951443359255791, -0.009289800189435482, -0.0032802110072225332, 0.016323186457157135, 0.01783842407166958, -0.0945175290107727, 0.033388473093509674, -0.07169798016548157, 0.06039775535464287, -0.07363878190517426, -0.05581212043762207, -0.0016216334188356996, -0.06260178238153458, 0.017481667920947075, 0.07375195622444153, 0.03267374634742737, -0.04823506250977516, -0.021072249859571457, -0.06121167540550232, 0.024473506957292557, 0.037952128797769547, 0.0360078439116478, 0.0491141676902771, 0.06194033473730087, -0.06165227293968201, 0.016976216807961464, 0.039694931358098984, -0.017313724383711815, -0.024048879742622375,

## Step 4 — Store in Chroma
Load the shelves: put each chunk's text + vector + source into a persistent Chroma collection so you don't have to re-embed every time.

**Hints:**
- `client = chromadb.PersistentClient(path='../../data/chroma_kb')` — saves to disk so it survives a kernel restart.
- `collection = client.get_or_create_collection('security_kb')`.
- `collection.add(ids=chunk_ids, documents=chunk_texts, embeddings=embeddings, metadatas=[{'source': s} for s in chunk_sources])`.
- Note: we pass our OWN embeddings on purpose — Chroma can embed for you, but the point of this task is that *you* own the embedding step.
- Re-running `.add` with the same ids errors; if you re-run, either delete the collection first or use `.upsert`. Confirm with `collection.count()`.

In [ ]:
# TODO: create a PersistentClient + collection; add ids/documents/embeddings/metadatas; print collection.count()
client=chromadb.PersistentClient('/Users/rohanb/06_projects/CYBERSECURITY/data/chroma_kb')
collection = client.get_or_create_collection('security_kb')
collection.add(ids=chunk_ids, 
               documents=chunk_texts, 
               embeddings=embeddings, 
               metadatas=[{'source':s} for s in chunk_sources])
collection.count()

20

## Step 5 — Sanity check (a sneak peek at retrieval)
You don't build real retrieval until Task 2, but let's prove the store works. Ask it a question *in your own words* and see if the right chunk comes back.

**Hints:**
- Embed a query the SAME way: `q = model.encode(['an attacker is probing lots of ports on my server']).tolist()`.
- `collection.query(query_embeddings=q, n_results=2)` — look at the returned `documents` and `metadatas`.
- **The test:** did the top result come from `port_scan`? If a question with *no shared keywords* still pulls the right doc, semantic search is working — that's the magic.

In [23]:
# TODO: embed a test question; collection.query(...); print the top chunk + its source
q=model.encode(['an attacker is probing lots of ports on my server']).tolist()
collection.query(query_embeddings=q, n_results=2)

{'ids': [['port_scan_1', 'port_scan_4']],
 'embeddings': None,
 'documents': [['What it is\nA port scan is a reconnaissance technique where an attacker sends probes to many ports on a target host to discover which services are running and reachable. It is usually the first stage of an attack — the attacker is mapping the attack surface before choosing an exploit. Common tools include nmap and masscan. In the MITRE ATT&CK framework this maps to T1046 (Network Service Discovery).',
   'Analyst response / playbook\n1. Confirm the source IP and whether it is internal or external.\n2. Identify which ports/services were probed and whether any are sensitive (RDP, SSH, SMB, databases).\n3. Check whether the scan was followed by a connection to an open service (possible exploitation attempt).\n4. If internal source: treat as possible lateral movement or a compromised host — escalate.\n5. If external and noisy/broad: likely opportunistic internet background noise — rate-limit or block at the fir

---
### ✅ You pass RAG Task 1 when:
1. You loaded the 4 KB docs and split them into coherent, single-topic **chunks**.
2. You **embedded** every chunk with sentence-transformers (384-dim vectors) and can explain what an embedding *is*.
3. You **stored** them in a persistent Chroma collection (`collection.count()` matches your chunk count).
4. Your Step 5 test question pulls back the *right* chunk even without shared keywords — and you can say why that works.

Paste me your chunk count + the Step 5 result and I'll review before we build Task 2 (real retrieval).